In [1]:
REPO = '../'
RESULT_TABLE = f'{REPO}/data/table'
RESULT_OBJ = f'{REPO}/data/object'
FIGURE_FOLDER= f'{REPO}/data/figure'
SETTING_FOLDER = f'{REPO}/data/setting'
EXTERNAL_DATA=f'{REPO}/data/external'
# load Terra API for get data table from Terra
import sys
sys.path.append(REPO)
from settings import COLOR_PAlETTE
from utils.visual import *

import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scPipe as sp
import scanpy as sc
import seaborn as sns
import warnings
import commentjson
### Additional Colormap
with open(f'{SETTING_FOLDER}/ColorMap.json','r') as f:
    colormap = commentjson.load(f)
COLOR_PAlETTE.update(colormap)
plt.style.use(f'{REPO}/paper.mplstyle')
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [2]:
obs_path = f'{RESULT_TABLE}/TumorRandom.csv'
adata = sc.read(f'{RESULT_OBJ}/gex_all.h5ad')
obs = pd.read_csv(obs_path,index_col=0)
adata = adata[adata.obs.index.isin(obs.index),:]
print(f"Detect N={adata.shape[0]:,} tumor cells with high quality GEX profiles;")
print(f"max(% of mitocondrial reads): {adata.obs['pct_counts_mito'].max():.0f}")
sample_meta =  pd.read_excel(f'{RESULT_TABLE}/Supplementary Table 1.xlsx',index_col=0).replace(np.nan,'N/A')

Detect N=65,206 tumor cells with high quality GEX profiles;
max(% of mitocondrial reads): 5


# Detect the robust recurrent program robustRPH

- Run the workflow https://app.terra.bio/#workspaces/vanallen-firecloud-nih/HR_BrCa_16-466/workflows/jf_scpipe_rna/GEX_CohortSet_DetectRHP
    - Here, we directly use the tumor h5ad as input
- Get the basis and coef pickle files.

In [3]:
import pickle
POPULATION = 'Tumor_Split_1'
with open(f'{RESULT_TABLE}/MPs/{POPULATION}/nmf_basis.pickle', "rb") as input_file:
    programs_basis = pickle.load(input_file)

with open(f'{RESULT_TABLE}/MPs/{POPULATION}/nmf_coef.pickle', "rb") as input_file:
    programs_coef = pickle.load(input_file)
n_programs = programs_basis[list(programs_basis.keys())[2]].shape[1] * len(programs_basis)
with open(f'{RESULT_TABLE}/MPs/{POPULATION}/robustRPH.pickle', "rb") as input_file:
    sp1_robustRPH = pickle.load(input_file)
sp1_robustRPH.columns = 'SP1_'+sp1_robustRPH.columns 
sp1_programs_basis = {'SP1_'+k:v.rename(columns=dict(zip(v.columns,'SP1_'+v.columns))) for k,v in programs_basis.items()}
print(f"Out of {n_programs:,} detected programs, {sp1_robustRPH.shape[1]} programs are robust for SPLIT1.")

POPULATION = 'Tumor_Split_2'
with open(f'{RESULT_TABLE}/MPs/{POPULATION}/nmf_basis.pickle', "rb") as input_file:
    programs_basis = pickle.load(input_file)

with open(f'{RESULT_TABLE}/MPs/{POPULATION}/nmf_coef.pickle', "rb") as input_file:
    programs_coef = pickle.load(input_file)
n_programs = programs_basis[list(programs_basis.keys())[2]].shape[1] * len(programs_basis)
with open(f'{RESULT_TABLE}/MPs/{POPULATION}/robustRPH.pickle', "rb") as input_file:
    sp2_robustRPH = pickle.load(input_file)
sp2_robustRPH.columns = 'SP2_'+sp2_robustRPH.columns 
sp2_programs_basis = {'SP2_'+k:v.rename(columns=dict(zip(v.columns,'SP2_'+v.columns))) for k,v in programs_basis.items()}
print(f"Out of {n_programs:,} detected programs, {sp2_robustRPH.shape[1]} programs are robust for SPLIT2.")

robustRPH = pd.concat([sp1_robustRPH,sp2_robustRPH],axis=1)
programs_basis = sp1_programs_basis
programs_basis.update(sp2_programs_basis)


Out of 1,326 detected programs, 171 programs are robust for SPLIT1.
Out of 1,326 detected programs, 175 programs are robust for SPLIT2.


## Get a list of genes in the analysis

In [4]:
bg_genes = pd.Index([])
for k,v in programs_basis.items():
    bg_genes = v.index.union(bg_genes)

### Perform the clustering

as shown above, in timepoints Baseline, W3D1, W7D1, and AfterSurg, we have at least `2` samples per timepoints. Thus, we aimed to find metaprograms that at least shared among three samples and set the `Min_group_size` parameter to be `2`.

In [5]:
import palettable
Cluster_Map,MP_Genesets,Programs_Order,MP_colors = sp.ext.clusterRobustRPH(
    robustRPH=robustRPH,
    programs_basis=programs_basis,
    Min_group_size=2,
    palette=palettable.tableau.Tableau_20.hex_colors + palettable.tableau.TrafficLight_9.hex_colors
)

[Processing]Cluster MP1 has 63 RPH programs.
[Processing]Cluster MP2 has 46 RPH programs.
[Processing]Cluster MP3 has 27 RPH programs.
[Processing]Cluster MP4 has 7 RPH programs.
[Processing]Cluster MP5 has 12 RPH programs.
[Processing]Cluster MP6 has 9 RPH programs.
[Processing]Cluster MP7 has 9 RPH programs.
[Processing]Cluster MP8 has 14 RPH programs.
[Processing]Cluster MP9 has 10 RPH programs.
[Processing]Cluster MP10 has 4 RPH programs.
[Processing]Cluster MP11 has 7 RPH programs.
[Processing]Cluster MP12 has 6 RPH programs.
[Processing]Cluster MP13 has 6 RPH programs.
[Processing]Cluster MP14 has 4 RPH programs.
[Processing]Cluster MP15 has 6 RPH programs.
[Processing]Cluster MP16 has 5 RPH programs.
[Processing]Cluster MP17 has 5 RPH programs.
[Processing]Cluster MP18 has 4 RPH programs.
[Processing]Cluster MP19 has 4 RPH programs.
[Processing]Cluster MP20 has 4 RPH programs.
[Processing]Cluster MP21 has 6 RPH programs.
[Processing]Cluster MP22 has 4 RPH programs.
[Processing]C

### Look at MPs related to low data quality 

In [6]:
display(pd.concat([
    MP_Genesets.apply(lambda c:c.str.startswith('MT-').sum(),axis=0).rename('Mitochondrial'),
    MP_Genesets.apply(lambda c:c.str.contains('^RP[LS]').sum(),axis=0).rename('Ribosomal')
],axis=1).sort_values(['Mitochondrial','Ribosomal'],ascending=False))

low_quality_mps = ['MP1']

,Mitochondrial,Ribosomal
MP1,10,34
MP14,1,11
MP11,1,0
MP2,0,0
MP3,0,0
MP4,0,0
MP5,0,0
MP6,0,0
MP7,0,0
MP8,0,0


### Merge sample metadata into the clustering result

In [7]:
anno = Cluster_Map.rename('MetaProgram').to_frame()
anno['Sample'] = anno.index.map(lambda x: x.split('.')[0])
anno = anno.reset_index().merge(sample_meta,left_on='Sample',right_on='CCG_ID',how='left').set_index('index')

### explore the patient specific programs

In [8]:
pt_specific = anno.groupby(['MetaProgram'])['Patient'].nunique().sort_values()
pt_specific = pt_specific[pt_specific ==1].index.tolist()

In [9]:
anno['MetaProgram'].value_counts()[pt_specific]

Series([], Name: MetaProgram, dtype: int64)

In [10]:
anno.loc[anno['MetaProgram'].isin(pt_specific),:]

,MetaProgram,Sample,CCG_ID,BluePrint,Batch,AC_Treatment,Tech,WES_Profile,BulkRNA_Profile,PAM50,BestResponse,Timepoint,stage,Patient,Treatment_Arm
index,,,,,,,,,,,,,,,


###  Calculate the complexity correaltion

In [11]:
nmf_corr_complexity = []
for k,v in programs_coef.items():
    tmp = v.corrwith(adata.obs.loc[v.index,'n_genes_by_counts'])
    nmf_corr_complexity.append(tmp)
del tmp
nmf_corr_complexity=pd.concat(nmf_corr_complexity,axis=0)
dt = pd.concat([nmf_corr_complexity.abs()>.4,Cluster_Map],axis=1,join='inner').groupby(1)[0].mean()
high_complexity = dt[dt>.5].index.tolist()
high_complexity

[]

In [12]:
dt[dt>.5]

Series([], Name: 0, dtype: float64)

### Remove patient specific, high complexity, and unknown program

In [13]:
anno = anno.loc[~anno['MetaProgram'].isin(pt_specific+low_quality_mps+['MP_Unknown']+high_complexity),:]
New_Programs_Order = [ x for x in Programs_Order if x in anno.index]
New_MP_Genesets = MP_Genesets[anno['MetaProgram'].unique().tolist()]
f"Generated {anno['MetaProgram'].nunique()} MP clusters which covered {(anno.shape[0]/robustRPH.shape[1]):.0%} robust RPH programs."

'Generated 28 MP clusters which covered 67% robust RPH programs.'

## Write out the Genesets for every MP

In [14]:
mp_gmt_path = f'{RESULT_TABLE}/MPs/SPLIT_MP_Programs.gmt'
with open(mp_gmt_path,'w') as f:
    for mp in New_MP_Genesets:
        f.write('\t'.join([mp,'MetaProgram_HRpos']+MP_Genesets[mp].tolist())+'\n')
